- 데이터 확인
- 중복 데이터 확인
- 이상 데이터 확인
- 손상된 알약을 가진 이미지가 있을까??
- 이미지마다 알약은 몇개씩 있을까??
- 

## 데이터 로드

JSON 라벨 파일 하나와 연결된 이미지 한 장을 읽어 확인한다.
JSON은 이미지 파일명, 알약 위치(bbox), 종류 ID를 담은 라벨 파일이다.
아래 셀을 위에서부터 실행하면 `img`에 RGB 이미지, `annotations`에 라벨이 저장된다.
전체 데이터 로드는 아니며, 다른 JSON에 나뉜 같은 이미지의 라벨은 아직 합치지 않는다.

In [10]:
import json
from pathlib import Path

import cv2
import numpy as np

# 1. 프로젝트 최상위 폴더를 찾는다.
# 현재 폴더부터 상위 폴더까지 확인한다.
# src/config.py와 pyproject.toml이 함께 있는 곳을 프로젝트 루트로 판단한다.
PROJECT_ROOT = next(
    (
        folder
        for folder in (Path.cwd(), *Path.cwd().parents)
        if (folder / "src" / "config.py").is_file() and (folder / "pyproject.toml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("pill-bbox 프로젝트 안에서 노트북을 실행해 주세요.")

# 2. JSON 라벨 폴더와 이미지 폴더의 경로를 만든다.
annotation_dir = PROJECT_ROOT / "data" / "train_annotations"
image_dir = PROJECT_ROOT / "data" / "train_images"

# 3. 하위 폴더까지 탐색해서 JSON 파일 하나를 선택한다.
annotation_path = next(annotation_dir.rglob("*.json"), None)
if annotation_path is None:
    raise FileNotFoundError(f"JSON 라벨 파일이 없습니다: {annotation_dir}")

# 4. JSON 파일을 읽어 파이썬 딕셔너리로 바꾼다.
with annotation_path.open(encoding="utf-8") as file:
    data = json.load(file)

# 5. JSON에 기록된 첫 번째 이미지의 정보를 가져온다.
image_info = data["images"][0]

# 이미지 폴더 경로와 파일명을 합쳐 실제 이미지 경로를 만든다.
image_path = image_dir / image_info["file_name"]

# 6. 이미지 파일을 바이트 배열로 읽고, 이미지 배열로 변환한다.
# 이 방식은 한글이 포함된 경로에서도 이미지를 읽을 수 있게 한다.
img = cv2.imdecode(
    np.fromfile(image_path, dtype=np.uint8),
    cv2.IMREAD_COLOR,
)

# 파일을 이미지로 해석하지 못했다면 오류를 발생시킨다.
if img is None:
    raise ValueError(f"이미지를 읽을 수 없습니다: {image_path}")

# OpenCV 기본 색상 순서인 BGR을 RGB로 바꾼다.
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 7. JSON의 라벨 중 선택한 이미지에 연결된 것만 모은다.
annotations = [
    annotation for annotation in data["annotations"] if annotation["image_id"] == image_info["id"]
]

# 8. 읽은 결과를 확인한다.
print("이미지:", image_path.name)
print("이미지 크기:", img.shape)

for annotation in annotations:
    print("bbox:", annotation["bbox"])
    print("클래스:", annotation["category_id"])

이미지: K-001900-016548-019607-029451_0_2_0_2_70_000_200.png
이미지 크기: (1280, 976, 3)
bbox: [644, 845, 189, 190]
클래스: 1900
